## Quick Exploratory Data Analysis

### 1. Importar

In [23]:
import pandas as pd
import json

### 2. Load

In [44]:
categories = pd.read_csv("../data/desafio_tecnico_ai_eng - categories.csv")
customers = pd.read_csv("../data/desafio_tecnico_ai_eng - customers.csv")
products = pd.read_csv("../data/desafio_tecnico_ai_eng - products.csv")
orders = pd.read_csv("../data/desafio_tecnico_ai_eng - orders.csv")
order_items = pd.read_csv("../data/desafio_tecnico_ai_eng - order_items.csv")
promotions = pd.read_csv("../data/desafio_tecnico_ai_eng - promotions.csv")

In [38]:
categories.head(1)

,category_id,name,description
0,1,Guitarras,"Guitarras elétricas de todos os estilos, do ro..."


In [39]:
customers.head(1)

,customer_id,name,phone,email,city
0,1,Lucas Mendes da Silva,(67) 99812-3456,lucas.mendes@jmail.com,Campo Grande


In [46]:
products.head(1)

,product_id,price_brl,name,category_id,description,stock_quantity,status,specs,created_at
0,81,599.9,Yamaha C40 Nylon Natural,5,Violão clássico com tampo em Spruce e fundo/la...,12,active,"{""top"":""Spruce"",""back_sides"":""Meranti"",""neck"":...",2021-03-15


In [49]:
products.loc[products['product_id'] == 81, 'specs'].values[0]

'{"top":"Spruce","back_sides":"Meranti","neck":"Nato","strings":"nylon","scale":"650mm","electronics":"no","color":"Natural"}'

In [50]:
products.loc[products['product_id'] == 81, 'description'].values[0]

'Violão clássico com tampo em Spruce e fundo/laterais em Meranti. Ideal para estudantes e iniciantes, com excelente projeção sonora.'

In [41]:
orders.head(1)

,order_id,customer_id,order_date,status,total_brl,payment_method,tracking_code,estimated_delivery,notes
0,1,3,2025-10-15,delivered,11499.0,pix,BRAB1234567BR,2025-10-25,NaN


In [58]:
orders['status'].unique()

array(['delivered', 'shipped', 'confirmed', 'pending', 'cancelled'],
      dtype=object)

In [59]:
orders['notes'].unique()

array([nan, 'Pedido cancelado pelo cliente antes do processamento',
       'Item fora de estoque no momento do pedido',
       'Pagamento não confirmado dentro do prazo',
       'Endereço de entrega inválido informado pelo cliente'],
      dtype=object)

In [42]:
order_items.head(1)

,order_id,quantity,product_id
0,1,1,93


In [43]:
promotions.head(1)

,promotion_id,product_id,discount_percent,description,is_active
0,1,102,10,Black Friday,0


In [55]:
promotions['description'].unique()

array(['Black Friday', 'Liquidação de Inverno', 'Queima de Estoque',
       'Oferta Especial', 'Aniversário da Loja', 'Promoção de Páscoa',
       'Semana do Músico', 'Desconto Exclusivo'], dtype=object)

In [25]:
print("SHAPE DOS DATASETS")
for name, df in [("categories", categories), ("customers", customers),
                 ("products", products), ("orders", orders),
                 ("order_items", order_items), ("promotions", promotions)]:
    print(f"{name}: {df.shape[0]} linhas x {df.shape[1]} colunas")

SHAPE DOS DATASETS
categories: 9 linhas x 3 colunas
customers: 50 linhas x 5 colunas
products: 65 linhas x 9 colunas
orders: 20 linhas x 9 colunas
order_items: 22 linhas x 3 colunas
promotions: 25 linhas x 5 colunas


## 3. Produtos por Categoria

In [26]:
merged = products.merge(categories, on="category_id", suffixes=("", "_cat"))
cat_summary = merged.groupby("name_cat").agg(
    qtd_produtos=("product_id", "count"),
    preco_min=("price_brl", "min"),
    preco_max=("price_brl", "max"),
    preco_medio=("price_brl", lambda x: round(x.mean(), 2)),
    estoque_total=("stock_quantity", "sum")
).sort_values("qtd_produtos", ascending=False)

print("\n=== PRODUTOS POR CATEGORIA ===")
print(cat_summary)

print("\n Categorias SEM produtos:",
      list(set(categories["name"]) - set(merged["name_cat"])))


=== PRODUTOS POR CATEGORIA ===
                      qtd_produtos  preco_min  preco_max  preco_medio  \
name_cat                                                                
Violões                         35      429.9    12000.0      2459.47   
Ukuleles                        15      159.9     1899.0       734.69   
Guitarras                        5     5400.0    18900.0     10220.00   
Baixos                           4      441.0    11445.0      6943.75   
Baterias e Percussão             3     4439.0    14922.0     10882.00   
Teclados e Pianos                3     1127.0    19567.0     10015.00   

                      estoque_total  
name_cat                             
Violões                         198  
Ukuleles                         99  
Guitarras                        42  
Baixos                           42  
Baterias e Percussão             23  
Teclados e Pianos                13  

 Categorias SEM produtos: ['Instrumentos de Sopro (Madeiras)', 'Instrumentos d

## 4. Status dos Produtos

In [37]:
products.head(1)

,product_id,price_brl,name,category_id,description,stock_quantity,status,specs,created_at
0,81,599.9,Yamaha C40 Nylon Natural,5,Violão clássico com tampo em Spruce e fundo/la...,12,active,"{""top"":""Spruce"",""back_sides"":""Meranti"",""neck"":...",2021-03-15


In [28]:
print(products["status"].value_counts())

status
active          62
discontinued     2
coming_soon      1
Name: count, dtype: int64


## 4.1 Pontos de Atenção - Estoque zero mas status active

In [29]:
zero_stock_active = products[(products["stock_quantity"] == 0) & (products["status"] == "active")]
print(f"\n4.1 - Estoque ZERO com status 'active' ({len(zero_stock_active)} produtos):")
print(zero_stock_active[["product_id", "name", "stock_quantity", "status"]].to_string(index=False))
print("Tratar como 'temporariamente indisponível' na tool")


4.1 - Estoque ZERO com status 'active' (1 produtos):
 product_id                                name  stock_quantity status
         96 Giannini GF-3D Dreadnought Sunburst               0 active
Tratar como 'temporariamente indisponível' na tool


## 4.2 Pontos de Atenção - Produtos descontinuados

In [30]:
discontinued = products[products["status"] == "discontinued"]
print(f"\n4.2 - Produtos DESCONTINUADOS ({len(discontinued)}):")
print(discontinued[["product_id", "name", "status"]].to_string(index=False))
print("Não mostrar como opção de compra, sugerir alternativa")


4.2 - Produtos DESCONTINUADOS (2):
 product_id                                name       status
        113 Shelby SN-7C 7 Cordas Nylon Natural discontinued
        136        Ibanez Bass 2X Electric Bass discontinued
Não mostrar como opção de compra, sugerir alternativa


## 4.3 Pontos de Atenção - Produtos coming_soon

In [31]:
coming = products[products["status"] == "coming_soon"]
print(f"\n4.3 - Produtos COMING SOON ({len(coming)}):")
print(coming[["product_id", "name", "status"]].to_string(index=False))
print("Informar que está chegando, sem permitir compra")


4.3 - Produtos COMING SOON (1):
 product_id                           name      status
        130 Kala KA-ABP-CTG Barítono Cedar coming_soon
Informar que está chegando, sem permitir compra


## 4.4 Pontos de Atenção - Promoções ativas vs expiradas

In [32]:
active_promos = promotions[promotions["is_active"] == 1]
inactive_promos = promotions[promotions["is_active"] == 0]

print(f"\n4.4 - Promoções: {len(active_promos)} ativas / {len(inactive_promos)} expiradas")
print("Promoções ATIVAS:")
active_detail = active_promos.merge(products[["product_id", "name", "price_brl"]], on="product_id")
active_detail["preco_com_desconto"] = round(active_detail["price_brl"] * (1 - active_detail["discount_percent"]/100), 2)
print(active_detail[["promotion_id", "name", "price_brl", "discount_percent", "preco_com_desconto", "description"]].to_string(index=False))
print("Promoções expiradas: informar que não está mais vigente e dar preço atual")


4.4 - Promoções: 4 ativas / 21 expiradas
Promoções ATIVAS:
 promotion_id                            name  price_brl  discount_percent  preco_com_desconto        description
           12   Kalani KAL-700T Tenor Natural      359.0                10              323.10    Oferta Especial
           14     Ohana CK-20 Concert Natural      549.0                20              439.20 Desconto Exclusivo
           19 Crafter HT-100 Folk Aço Natural     2399.0                18             1967.18 Desconto Exclusivo
           24 Taylor 110e Dreadnought Natural     5999.0                 8             5519.08 Desconto Exclusivo
Promoções expiradas: informar que não está mais vigente e dar preço atual


## 4.5 Pontos de Atenção - Inconsistência pedido x produto coming_soon

In [33]:
print("\n4.5 - Inconsistência: pedido com produto coming_soon")
order3_items = order_items[order_items["order_id"] == 3]
order3_products = order3_items.merge(products[["product_id", "name", "status"]], on="product_id")
print(order3_products[["order_id", "product_id", "quantity", "name", "status"]].to_string(index=False))
print("Dados com inconsistência pontual, manter como fornecido")


4.5 - Inconsistência: pedido com produto coming_soon
 order_id  product_id  quantity                           name      status
        3         130         2 Kala KA-ABP-CTG Barítono Cedar coming_soon
Dados com inconsistência pontual, manter como fornecido


In [34]:
# ressalto que no ponto acima o pedido pode ter sido feito e o cliente está ciente que ele está 'coming soon'

## 5. Pedidos 

In [35]:
print(f"Total: {len(orders)}")
print(orders["status"].value_counts())
print(f"\nMétodos de pagamento:")
print(orders["payment_method"].value_counts())

Total: 20
status
delivered    7
confirmed    4
pending      4
cancelled    4
shipped      1
Name: count, dtype: int64

Métodos de pagamento:
payment_method
pix           5
credit_12x    4
credit_6x     3
boleto        3
credit_3x     3
debit         2
Name: count, dtype: int64


## 6. Clientes

In [36]:
print("\n=== CLIENTES ===")
print(f"Total: {len(customers)}")
print(f"Cidades:")
print(customers["city"].value_counts())


=== CLIENTES ===
Total: 50
Cidades:
city
Campo Grande    24
Dourados         7
Três Lagoas      7
Corumbá          6
Ponta Porã       6
Name: count, dtype: int64
